In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('faceit_clean.csv')

# Preparing features

In [2]:
label_cols = ['team1_win', 'score_diff']
cat_cols = ['map']
num_cols = [c for c in df.columns if c not in label_cols + cat_cols]

X = df[num_cols + cat_cols]
y = df['score_diff']

# Train / Test Split

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# Preprocessing Pipeline

In [4]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols),
    ]
)

# Linear Regression

In [5]:
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

lr_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

lr_pipe.fit(X_train, y_train)
y_pred_lr = lr_pipe.predict(X_test)

print('Linear Regression:')
print(f'MAE:  {mean_absolute_error(y_test, y_pred_lr):.3f}')
print(f'RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_lr)):.3f}')
print(f'R2:   {r2_score(y_test, y_pred_lr):.3f}')

Linear Regression:
MAE:  4.016
RMSE: 4.912
R2:   0.292


# Random Forest

In [6]:
from sklearn.ensemble import RandomForestRegressor

rf_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42))
])

rf_pipe.fit(X_train, y_train)
y_pred_rf = rf_pipe.predict(X_test)

print('Random Forest Regressor:')
print(f'MAE:  {mean_absolute_error(y_test, y_pred_rf):.3f}')
print(f'RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_rf)):.3f}')
print(f'R2:   {r2_score(y_test, y_pred_rf):.3f}')

Random Forest Regressor:
MAE:  3.748
RMSE: 4.670
R2:   0.360


# Neural Network

In [11]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [17]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation, Dropout
from tensorflow.keras.callbacks import EarlyStopping

input_dim = X_train_processed.shape[1]

model = Sequential()

model.add(Dense(128, input_shape=(input_dim,)))
model.add(Activation('relu'))
model.add(Dropout(0.3))
model.add(Dense(64))
model.add(Activation('relu'))
model.add(Dropout(0.3))
model.add(Dense(32))
model.add(Activation('relu'))
model.add(Dropout(0.2))
model.add(Dense(1))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [18]:
model.compile(loss='mean_squared_error',
optimizer='adam',
metrics=['mae'])

In [19]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=30,
    restore_best_weights=True
)

In [20]:
model.fit(
    X_train_processed, y_train,
    epochs=600,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/600
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 30.6593 - mae: 4.7637 - val_loss: 24.4375 - val_mae: 4.0749
Epoch 2/600
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 25.8969 - mae: 4.2098 - val_loss: 22.6109 - val_mae: 3.8308
Epoch 3/600
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 24.4907 - mae: 4.0313 - val_loss: 22.5133 - val_mae: 3.7904
Epoch 4/600
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 24.0417 - mae: 3.9879 - val_loss: 21.9230 - val_mae: 3.7326
Epoch 5/600
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 23.4838 - mae: 3.9182 - val_loss: 21.7080 - val_mae: 3.7054
Epoch 6/600
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 23.5898 - mae: 3.9401 - val_loss: 21.8049 - val_mae: 3.7209
Epoch 7/600
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 23.5411 - mae: 3.9124 - val_loss: 21.5668 - val_mae: 3.6921
Epoch 8/600
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 22.9819 - mae: 3.8890 - val_loss: 21.9743 - val_mae: 3.7391
Epoch 9/600
100/100 ━━━━━━━━━━━━

In [21]:
y_pred_nn = model.predict(X_test_processed).flatten()

print('Neural Network:')
print(f'MAE:  {mean_absolute_error(y_test, y_pred_nn):.3f}')
print(f'RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_nn)):.3f}')
print(f'R2:   {r2_score(y_test, y_pred_nn):.3f}')

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step  
Neural Network:
MAE:  3.767
RMSE: 4.721
R2:   0.346


# Evaluation

In [23]:
results = {
    'Linear Regression': {
        'MAE': mean_absolute_error(y_test, y_pred_lr),
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred_lr)),
        'R2': r2_score(y_test, y_pred_lr),
    },
    'Random Forest': {
        'MAE': mean_absolute_error(y_test, y_pred_rf),
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred_rf)),
        'R2': r2_score(y_test, y_pred_rf),
    },
    'Neural Network': {
        'MAE': mean_absolute_error(y_test, y_pred_nn),
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred_nn)),
        'R2': r2_score(y_test, y_pred_nn),
    }
}

results_df = pd.DataFrame(results).T
results_df = results_df.round(3)
print(results_df.to_string())


                     MAE   RMSE     R2
Linear Regression  4.016  4.912  0.292
Random Forest      3.748  4.670  0.360
Neural Network     3.767  4.721  0.346
